<div dir="rtl" align="right">

# درخت تصمیم برای رگرسیون

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
درخت تصمیم مثل بازی «بیست سؤالی» است: «متراژ کمتر از ۸۰ است؟ بله ← سن بنا بیشتر از ۲۰ است؟ ...». در هر **برگ**، پیش‌بینی برابر **میانگین** $y$ نمونه‌هایی است که به آن برگ رسیده‌اند. نتیجه یک تابع **پله‌ای** است.

> [یک درخت رگرسیون با عمق ۲. هر گره یک پرسش است؛ value میانگین آن ناحیه است.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/tree_reg_struct.png)

*یک درخت رگرسیون با عمق ۲. هر گره یک پرسش است؛ value میانگین آن ناحیه است.*

## درخت چطور سؤال را انتخاب می‌کند؟
برای هر ویژگی و هر نقطه برش ممکن $t$، داده را به دو گروه چپ ($x\le t$) و راست تقسیم می‌کند و **کاهش مجموع مربعات خطا** را حساب می‌کند. بهترین برش، کمترین مقدار این را دارد:

$$\text{SSE}=\sum_{i\in L}(y_i-\bar y_L)^2+\sum_{i\in R}(y_i-\bar y_R)^2$$

سپس همین کار را برای هر گروه تکرار می‌کند (**تقسیم بازگشتی**) تا به شرط توقف برسد.

## مثال عددی یک برش
$x=[1,2,3,4]$ و $y=[1,1,5,6]$:

| برش | گروه چپ | گروه راست | SSE |
|---|---|---|---|
| $x\le1.5$ | [1] | [1, 5, 6] | $0+14=14$ |
| $x\le2.5$ | [1, 1] | [5, 6] | $0+0.5=0.5$ |
| $x\le3.5$ | [1, 1, 5] | [6] | $10.67+0=10.67$ |

بهترین برش $x\le2.5$ است: چپ پیش‌بینی ۱ و راست ۵.۵.



</div>

In [ ]:
import numpy as np

# Find the best cut by trying every possible threshold
x = np.array([1, 2, 3, 4]); y = np.array([1, 1, 5, 6])
for t in [1.5, 2.5, 3.5]:
    L, R = y[x <= t], y[x > t]
    sse = np.sum((L - L.mean()) ** 2) + np.sum((R - R.mean()) ** 2)
    print(f"x <= {t}: left {L}, right {R}, SSE = {sse:.2f}")

<div dir="rtl" align="right">



## عمق درخت و بیش‌برازش
> [عمق ۱: فقط دو پله. عمق ۳: خوب. بدون محدودیت: از هر نقطه می‌گذرد (بیش‌برازش).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/tree_reg_depth.png)

*عمق ۱: فقط دو پله. عمق ۳: خوب. بدون محدودیت: از هر نقطه می‌گذرد (بیش‌برازش).*

[[lab:reg_tree]]

| هایپرپارامتر | اثر |
|---|---|
| `max_depth` | حداکثر عمق؛ کمتر یعنی ساده‌تر |
| `min_samples_leaf` | حداقل نمونه در هر برگ؛ بیشتر یعنی صاف‌تر |
| `min_samples_split` | حداقل نمونه برای اینکه گره تقسیم شود |
| `ccp_alpha` | هرس با پیچیدگی؛ شاخه‌های کم‌فایده حذف می‌شوند |

## مزایا و معایب
| مزایا | معایب |
|---|---|
| قابل فهم و قابل رسم | ناپایدار: تغییر کوچک داده، درخت کاملاً متفاوت |
| نیاز به مقیاس‌بندی ندارد | خروجی پله‌ای، نه صاف |
| روابط غیرخطی و اثر متقابل را خودش پیدا می‌کند | به‌تنهایی زود بیش‌برازش می‌کند |
| ویژگی‌های عددی و دسته‌ای | برون‌یابی نمی‌کند |

> راه حل ناپایداری درخت، ترکیب درخت‌های زیاد است: جنگل تصادفی و بوستینگ (دو درس بعد).



</div>

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor, export_text

d = load_diabetes(as_frame=True)
for depth in [1, 2, 3, 4, 6, 10, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=depth, random_state=0), d.data, d.target, cv=5, scoring="r2")
    print(f"max_depth={depth}: R2 = {s.mean():.3f}")

tree = DecisionTreeRegressor(max_depth=2, random_state=0).fit(d.data, d.target)
print(export_text(tree, feature_names=list(d.data.columns)))

<div dir="rtl" align="right">

### تمرین

یک DecisionTreeRegressor با min_samples_leaf=20 روی داده دیابت بساز و R² اعتبارسنجی را با max_depth=None مقایسه کن.

</div>

In [ ]:
# Your answer here